# MAE monocyte input build

# 1. Load processed monocyte AnnData object

## Dependencies, paths, and parameters

In [ ]:
from pathlib import Path
import gc
import os

import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp

# Documented private input and output path variables.
# Private AnnData objects and metadata workbooks are not distributed.
analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

PROJECT_ROOT = resolve_analysis_path(
    os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai"))
)
RAW_ADATA_PATH = resolve_analysis_path(
    os.environ.get("MAE_MONOCYTE_H5AD", "adata_mono_MAE.h5ad")
)
HVG_ADATA_PATH = resolve_analysis_path(
    os.environ.get("MAE_HVG_H5AD", os.path.join("outputs", "ai", "adata_mono_MAE_HVG5000.h5ad"))
)
METADATA_PATH = resolve_analysis_path(
    os.environ.get("MAE_METADATA_XLSX", "Metadata.xlsx")
)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

DISEASE_COL = "disease"
PATIENT_COL = "patient_id"
SAMPLE_COL = "sample_id"
DISEASES = ["CAR-T_CRS", "COVID19", "SLE"]
N_HVG = 5000
ALPHA = 0.7

# 2. Balanced-consensus HVG5000 construction

In [ ]:
adata = sc.read_h5ad(PROJECT_ROOT / "adata_mono_MAE.h5ad")
adata = adata[adata.obs[DISEASE_COL] != "HD", :].copy()

In [ ]:
# Disease-specific HVG tables 

hvg_tables = {}

for disease in DISEASES:

    print(f"\nProcessing: {disease}")

    mask = adata.obs[DISEASE_COL].astype(str).eq(disease)

    print(f"  Cells: {mask.sum():,}")
    print(f"  Genes: {adata.n_vars:,}")

    if mask.sum() == 0:
        raise ValueError(
            f"No cells found for disease label: {disease}"
        )

    # Create a VIEW rather than a full AnnData copy
    ad_disease = adata[mask, :]


    hvg = sc.pp.highly_variable_genes(
        ad_disease,
        flavor="seurat", # rather than 'seurat_v3' because we only have log1p-transformed data, not raw counts
        n_top_genes=None,  # not letting scanpy choose the number of HVGs, we will filter by dispersion below
        min_mean=-np.inf,
        max_mean=np.inf,
        min_disp=-np.inf,
        max_disp=np.inf,
        subset=False,
        inplace=False
    )

    # Make gene names explicit
    hvg.index = adata.var_names

    # Retain the statistics needed downstream
    hvg_tables[disease] = hvg[
        ["means", "dispersions", "dispersions_norm"]
    ].copy()

    del ad_disease
    gc.collect()

    print("  Done.")

In [ ]:
# build cross-disease ranking table
consensus = pd.DataFrame(index=adata.var_names)

for disease in DISEASES:

    stats = hvg_tables[disease]

    # Normalized dispersion
    disp = (
        stats["dispersions_norm"]
        .replace([np.inf, -np.inf], np.nan)
        .copy()
    )

    consensus[f"{disease}_dispersion_norm"] = disp

    # --------------------------------------------------------
    # Rank:
    # rank 1 = most variable gene
    #
    # NaN genes are explicitly assigned the WORST rank.
    # --------------------------------------------------------

    rank = disp.rank(
        ascending=False,
        method="average",
        na_option="keep"
    )

    n_valid = disp.notna().sum()

    rank = rank.fillna(n_valid + 1)

    consensus[f"{disease}_rank"] = rank

    # --------------------------------------------------------
    # Percentile:
    # 1 = most variable
    # ~0 = least variable
    # 0 = undefined / NaN dispersion
    #
    # IMPORTANT:
    # Keep NaNs during ranking and explicitly set them to zero.
    # --------------------------------------------------------

    pct = disp.rank(
        ascending=True,
        method="average",
        pct=True,
        na_option="keep"
    )

    pct = pct.fillna(0.0)

    consensus[f"{disease}_pct"] = pct

In [ ]:
# calculate balanced consensus score
pct_cols = [
    f"{disease}_pct"
    for disease in DISEASES
]

# Mean percentile:
# equal weight to CAR-T, COVID, and SLE
consensus["mean_pct"] = (
    consensus[pct_cols].mean(axis=1)
)

# Maximum percentile:
# preserves genes that are exceptionally variable in
# one disease, even if less variable in the other two
consensus["max_pct"] = (
    consensus[pct_cols].max(axis=1)
)

# Main balanced-consensus score
consensus["consensus_score"] = (
    ALPHA * consensus["mean_pct"]
    + (1 - ALPHA) * consensus["max_pct"]
)

# Rank all genes
consensus["consensus_rank"] = (
    consensus["consensus_score"]
    .rank(
        ascending=False,
        method="first"
    )
    .astype(int)
)

# Exactly 5000 genes
consensus["selected"] = (
    consensus["consensus_rank"] <= N_HVG
)

print(
    f"Selected HVGs: "
    f"{consensus['selected'].sum():,}"
)

In [ ]:
# sort results
consensus = consensus.sort_values(
    "consensus_rank"
)

consensus.head(20)

In [ ]:
# check if key genes remain
genes_check = [
    "IL1B",
    "S100A8",
    "S100A9",
    "CD14",
    "FCGR3A",
    "ISG15",
    "IFIT1",
    "IFIT2",
    "IFIT3",
    "IFI6",
    "IFI27",
    "IFITM1",
    "MX1",
    "OAS1",
    "OAS2"
]

genes_present = [
    g for g in genes_check
    if g in consensus.index
]

check_table = consensus.loc[
    genes_present,
    [
        *[f"{d}_rank" for d in DISEASES],
        *[f"{d}_pct" for d in DISEASES],
        "mean_pct",
        "max_pct",
        "consensus_score",
        "consensus_rank",
        "selected"
    ]
].sort_values("consensus_rank")

check_table

In [ ]:
# save the 5000 gene list and full audit table
hvg5000 = consensus.index[
    consensus["selected"]
].tolist()

print(len(hvg5000))
print(hvg5000[:20])


# Full auditable table
consensus.to_csv(
    "balanced_consensus_HVG5000_full_table.csv"
)

# Simple gene list
pd.Series(
    hvg5000,
    name="gene"
).to_csv(
    "balanced_consensus_HVG5000_genes.csv",
    index=False
)

# Restore original gene order when joining
consensus_original_order = consensus.reindex(
    adata.var_names
)

adata.var["hvg_ai"] = (
    consensus_original_order["selected"]
    .astype(bool)
)

adata.var["hvg_ai_rank"] = (
    consensus_original_order["consensus_rank"]
)

adata.var["hvg_ai_score"] = (
    consensus_original_order["consensus_score"]
)

print(
    adata.var["hvg_ai"].value_counts()
)

In [ ]:
# Diagnostic: composition of balanced-consensus HVGs

selected_genes = consensus.index[
    consensus["selected"]
]

# Disease-specific top-5000 sets
top_sets = {}

for disease in DISEASES:
    rank_col = f"{disease}_rank"

    top_sets[disease] = set(
        consensus
        .sort_values(rank_col)
        .head(N_HVG)
        .index
    )


# ------------------------------------------------------------
# 1. Overlap between final consensus set and each disease's
#    own top-5000 HVG set
# ------------------------------------------------------------

print("Overlap with disease-specific top-5000 HVGs:\n")

for disease in DISEASES:

    overlap = len(
        set(selected_genes) & top_sets[disease]
    )

    print(
        f"{disease}: "
        f"{overlap:,}/{N_HVG:,} "
        f"({overlap / N_HVG:.1%})"
    )

In [ ]:
# ============================================================
# 2. How many diseases support each selected HVG?
# ============================================================

support = pd.DataFrame(
    index=selected_genes
)

for disease in DISEASES:
    support[disease] = support.index.isin(
        top_sets[disease]
    )

support["n_diseases"] = (
    support[DISEASES]
    .sum(axis=1)
)

print("\nNumber of diseases in which each consensus HVG")
print("also belongs to the disease-specific top 5000:\n")

print(
    support["n_diseases"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# 3. Disease source of uniquely supported genes
# ============================================================

single_disease = support[
    support["n_diseases"] == 1
].copy()

single_disease["source"] = (
    single_disease[DISEASES]
    .idxmax(axis=1)
)

print("\nSingle-disease-supported genes:\n")

print(
    single_disease["source"]
    .value_counts()
)

In [ ]:
# freeze the gene list and create a new input object
HVG5000 = (
    consensus.loc[consensus["selected"]]
    .sort_values("consensus_rank")
    .index
    .tolist()
)

adata_m = adata[:, HVG5000].copy()

print(adata_m)
print(
    f"Cells: {adata_m.n_obs:,}\n"
    f"Genes: {adata_m.n_vars:,}"
)

In [ ]:
# save
adata_m.write_h5ad("adata_mono_MAE_HVG5000.h5ad")

# 3. Patient metadata and matrix verification

In [ ]:
adata_m = sc.read_h5ad("adata_mono_MAE_HVG5000.h5ad")

In [ ]:
print(adata_m)

print("\nX type:")
print(type(adata_m.X))

print("\nShape:")
print(adata_m.shape)

print("\nSparse:")
print(sp.issparse(adata_m.X))

print("\nDtype:")
print(adata_m.X.dtype)

print("\nAvailable metadata columns:")
print(adata_m.obs.columns.tolist())


In [ ]:
# create one patient_id column by matching metadata in 'Metadata.xlsx'
metadata = pd.read_excel("Metadata.xlsx")

# Keep only the columns needed and normalize values
metadata = metadata[["Patient_id", "Original_sample_id"]].copy()
metadata["Patient_id"] = metadata["Patient_id"].astype(str).str.strip()
metadata["Original_sample_id"] = metadata["Original_sample_id"].astype(str).str.strip()

# Drop duplicates so the mapping is unambiguous
metadata = metadata.dropna(subset=["Original_sample_id"]).drop_duplicates(subset=["Original_sample_id"])

# Build a sample_id -> patient_id mapping
sample_to_patient = metadata.set_index("Original_sample_id")["Patient_id"].to_dict()

# In this object, the available identifier column is sample_id
source_col = "sample_id" if "sample_id" in adata_m.obs.columns else "patient_id"

# Create the new patient_id column
adata_m.obs["patient_id"] = (
    adata_m.obs[source_col]
    .astype(str)
    .str.strip()
    .map(sample_to_patient)
)

adata_m.obs[["sample_id", "patient_id"]].head()

In [ ]:
# cells contributed by each patient
patient_cell_counts = (
    adata_m.obs
    .groupby(
        ['disease', 'patient_id'],
        observed=True
    )
    .size()
    .rename("n_cells")
    .reset_index()
)

print(
    patient_cell_counts
    .sort_values(
        ['disease', "n_cells"],
        ascending=[True, False]
    )
    .to_string(index=False)
)

In [ ]:
# expression value sanity check
if sp.issparse(adata_m.X):

    print("Minimum stored value:", adata_m.X.data.min())
    print("Maximum stored value:", adata_m.X.data.max())
    print("Mean stored non-zero value:", adata_m.X.data.mean())

    print(
        "Fraction non-zero:",
        adata_m.X.nnz /
        (adata_m.n_obs * adata_m.n_vars)
    )

else:

    print("Minimum:", adata_m.X.min())
    print("Maximum:", adata_m.X.max())
    print("Mean:", adata_m.X.mean())
    print(
        "Fraction non-zero:",
        np.count_nonzero(adata_m.X) / adata_m.X.size
    )